# Решения семинарских задач

Отдельный ноутбук с реализациями и проверками для семинарских задач.

In [ ]:
import torch
import numpy as np

print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())

## 1. Вычислительный граф

Реализация узлов вычислительного графа без сохранения промежуточных входов внутри слоёв.

In [ ]:
class Value:
    def __init__(self, data, requires_grad=False, _children=(), _backward=None, label='leaf'):
        self.data = torch.as_tensor(data, dtype=torch.float64)
        self.requires_grad = requires_grad
        self.grad = torch.zeros_like(self.data) if requires_grad else None
        self._prev = tuple(_children)
        self._backward = _backward
        self.label = label

    def backward(self, grad=None):
        if grad is None:
            if self.data.numel() != 1:
                raise RuntimeError('backward() scalar root required')
            grad = torch.ones_like(self.data)

        topo, seen = [], set()
        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._prev:
                    visit(parent)
                topo.append(node)
        visit(self)

        for node in topo:
            if node.requires_grad:
                node.grad = torch.zeros_like(node.data)
        self.grad = grad.clone() if self.grad is not None else grad

        for node in reversed(topo):
            if node._backward is not None:
                node._backward()

    def zero_grad(self):
        seen = set()
        def visit(node):
            if id(node) in seen:
                return
            seen.add(id(node))
            for parent in node._prev:
                visit(parent)
            if node.requires_grad:
                node.grad = torch.zeros_like(node.data)
        visit(self)


def unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(dim=0)
    for axis, size in enumerate(shape):
        if size == 1:
            grad = grad.sum(dim=axis, keepdim=True)
    return grad


class LinearGraph:
    def __init__(self, in_features, out_features, seed=0):
        gen = torch.Generator().manual_seed(seed)
        w = torch.randn(out_features, in_features, dtype=torch.float64, generator=gen) * (2 / in_features) ** 0.5
        b = torch.zeros(out_features, dtype=torch.float64)
        self.weight = Value(w, True, label='weight')
        self.bias = Value(b, True, label='bias')

    def forward(self, x):
        y = matmul(x, self.weight.t())
        return add(y, self.bias)


def matmul(a, b):
    result = Value(a.data @ b.data, a.requires_grad or b.requires_grad,
                   (a, b), label='matmul')
    def backward_fn():
        if a.requires_grad:
            a.grad = a.grad + result.grad @ b.data.t()
        if b.requires_grad:
            b.grad = b.grad + a.data.t() @ result.grad
    result._backward = backward_fn
    return result


def add(a, b):
    a = a if isinstance(a, Value) else Value(a)
    b = b if isinstance(b, Value) else Value(b)
    result = Value(a.data + b.data, a.requires_grad or b.requires_grad, (a, b), label='add')
    def backward_fn():
        if a.requires_grad:
            a.grad = a.grad + unbroadcast(result.grad, a.data.shape)
        if b.requires_grad:
            b.grad = b.grad + unbroadcast(result.grad, b.data.shape)
    result._backward = backward_fn
    return result


def relu(x):
    mask = x.data > 0
    result = Value(torch.where(mask, x.data, torch.zeros_like(x.data)),
                   x.requires_grad, (x,), label='relu')
    def backward_fn():
        if x.requires_grad:
            x.grad = x.grad + result.grad * mask
    result._backward = backward_fn
    return result


def mse(pred, target):
    diff = pred.data - target.data
    result = Value(diff.square().mean(), pred.requires_grad or target.requires_grad,
                   (pred, target), label='mse')
    def backward_fn():
        g = result.grad * (2 / diff.numel()) * diff
        if pred.requires_grad:
            pred.grad = pred.grad + g
        if target.requires_grad:
            target.grad = target.grad - g
    result._backward = backward_fn
    return result


def sum_value(x):
    result = Value(x.data.sum(), x.requires_grad, (x,), label='sum')
    def backward_fn():
        if x.requires_grad:
            x.grad = x.grad + torch.ones_like(x.data) * result.grad
    result._backward = backward_fn
    return result


def transpose(self):
    result = Value(self.data.t(), self.requires_grad, (self,), label='transpose')
    def backward_fn():
        if self.requires_grad:
            self.grad = self.grad + result.grad.t()
    result._backward = backward_fn
    return result

Value.t = transpose
Value.sum = sum_value


def check_graph():
    torch.manual_seed(11)
    x_np = torch.randn(4, 3, dtype=torch.float64)
    w_np = torch.randn(2, 3, dtype=torch.float64)
    b_np = torch.randn(2, dtype=torch.float64)
    y_np = torch.randn(4, 2, dtype=torch.float64)

    x = Value(x_np, True)
    w = Value(w_np, True)
    b = Value(b_np, True)
    pred = add(matmul(x, w.t()), b)
    loss = mse(pred, Value(y_np))
    loss.backward()

    x_t = x_np.clone().requires_grad_(True)
    w_t = w_np.clone().requires_grad_(True)
    b_t = b_np.clone().requires_grad_(True)
    pred_t = x_t @ w_t.t() + b_t
    loss_t = (pred_t - y_np).square().mean()
    loss_t.backward()

    print('forward match:', torch.allclose(pred.data, pred_t.detach(), atol=1e-12))
    print('x grad match:', torch.allclose(x.grad, x_t.grad, atol=1e-12, rtol=1e-9))
    print('w grad match:', torch.allclose(w.grad, w_t.grad, atol=1e-12, rtol=1e-9))
    print('b grad match:', torch.allclose(b.grad, b_t.grad, atol=1e-12, rtol=1e-9))

    # Два вызова до backward. Общий weight получает вклад обеих ветвей.
    shared_w = Value(w_np, True)
    shared_x = Value(x_np, True)
    first = matmul(shared_x, shared_w.t())
    second = matmul(shared_x, shared_w.t())
    branch_loss = mse(add(first, second), Value(torch.zeros_like(first.data)))
    branch_loss.backward()
    print('shared weight grad finite:', torch.isfinite(shared_w.grad).all().item())

    branch_loss.zero_grad()
    branch_loss.backward()
    print('zero_grad repeat finite:', torch.isfinite(shared_w.grad).all().item())


def numerical_gradients():
    x = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=torch.float64)
    w = torch.tensor([[0.7, -1.1], [1.3, 0.2]], dtype=torch.float64)
    y = torch.tensor([[1.0, 0.0], [-1.0, 2.0]], dtype=torch.float64)
    eps = 1e-6

    def scalar_loss(xv, wv):
        return ((xv @ wv.t() - y).pow(2).mean()).item()

    xg = torch.zeros_like(x)
    wg = torch.zeros_like(w)
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            xp, xm = x.clone(), x.clone()
            xp[i, j] += eps; xm[i, j] -= eps
            xg[i, j] = (scalar_loss(xp, w) - scalar_loss(xm, w)) / (2 * eps)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            wp, wm = w.clone(), w.clone()
            wp[i, j] += eps; wm[i, j] -= eps
            wg[i, j] = (scalar_loss(x, wp) - scalar_loss(x, wm)) / (2 * eps)

    xv = Value(x, True)
    wv = Value(w, True)
    mse(matmul(xv, wv.t()), Value(y)).backward()
    print('finite difference x max error:', (xv.grad - xg).abs().max().item())
    print('finite difference w max error:', (wv.grad - wg).abs().max().item())

check_graph()
numerical_gradients()

# Пример двух вызовов до backward без cached_x: каждый узел хранит собственный x.
cache_example_linear = LinearGraph(2, 1, seed=3)
x1 = Value(torch.tensor([[1.0, 2.0]], dtype=torch.float64), True)
x2 = Value(torch.tensor([[3.0, -1.0]], dtype=torch.float64), True)
out1 = cache_example_linear.forward(x1)
out2 = cache_example_linear.forward(x2)
mse(add(out1, out2), Value(torch.zeros((1, 1), dtype=torch.float64))).backward()
print('two calls: x1 and x2 gradients are finite:',
      torch.isfinite(x1.grad).all().item(), torch.isfinite(x2.grad).all().item())

## 2. Миниатюрный PyTorch

Ручные forward/backward на `torch.Tensor`, включая перенос параметров и градиентов на CPU/GPU.

In [ ]:
class MiniLinear:
    def __init__(self, in_features, out_features, device='cpu', dtype=torch.float32):
        self.weight = torch.randn(out_features, in_features, device=device, dtype=dtype,
                                  requires_grad=True)
        self.bias = torch.zeros(out_features, device=device, dtype=dtype, requires_grad=True)

    def forward(self, x):
        self._x = x
        return x @ self.weight.t() + self.bias

    def backward(self, grad):
        self.weight.grad = (grad.t() @ self._x).to(self.weight.device)
        self.bias.grad = grad.sum(dim=0).to(self.bias.device)
        return grad @ self.weight

    def parameters(self):
        return [self.weight, self.bias]

    def to(self, device):
        self.weight.data = self.weight.data.to(device)
        self.bias.data = self.bias.data.to(device)
        if self.weight.grad is not None:
            self.weight.grad = self.weight.grad.to(device)
        if self.bias.grad is not None:
            self.bias.grad = self.bias.grad.to(device)
        return self

class MiniReLU:
    def forward(self, x):
        self.mask = x > 0
        return torch.where(self.mask, x, torch.zeros_like(x))

    def backward(self, grad):
        return grad * self.mask

class MiniMSE:
    def forward(self, pred, target):
        self.diff = pred - target
        return self.diff.square().mean()

    def backward(self):
        return 2 * self.diff / self.diff.numel()

class MiniSGD:
    def __init__(self, parameters, lr=0.05):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for parameter in self.parameters:
            parameter.grad = None

    def step(self):
        with torch.no_grad():
            for parameter in self.parameters:
                parameter -= self.lr * parameter.grad


def train_minipy_torch(epochs=100):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(42)

    x = torch.randn(256, 2, dtype=torch.float32)
    y = (2 * x[:, 0] - 3 * x[:, 1] + 1).unsqueeze(1)
    x, y = x.to(device), y.to(device)

    l1 = MiniLinear(2, 100, device=device)
    relu = MiniReLU()
    l2 = MiniLinear(100, 1, device=device)
    parameters = l1.parameters() + l2.parameters()
    optimizer = MiniSGD(parameters, lr=0.01)
    mse = MiniMSE()

    losses = []
    for _ in range(epochs):
        optimizer.zero_grad()
        hidden = relu.forward(l1.forward(x))
        pred = l2.forward(hidden)
        loss = mse.forward(pred, y)
        grad = l2.backward(mse.backward())
        grad = relu.backward(grad)
        l1.backward(grad)
        optimizer.step()
        losses.append(loss.detach().item())

    l1.to('cpu')
    l2.to('cpu')
    print('device:', device)
    print('final loss:', losses[-1])
    print('model moved back to:', l1.weight.device)
    return losses

losses = train_minipy_torch()

In [ ]:
import time


def benchmark_minipy(torch_device, batch_size, warmup=5, repeats=20):
    torch.manual_seed(123)
    l1 = MiniLinear(2, 100, device=torch_device)
    relu = MiniReLU()
    l2 = MiniLinear(100, 1, device=torch_device)
    params = l1.parameters() + l2.parameters()
    optimizer = MiniSGD(params, lr=0.01)
    loss_fn = MiniMSE()

    x = torch.randn(batch_size, 2, device=torch_device)
    y = (2*x[:, 0] - 3*x[:, 1] + 1).unsqueeze(1)

    def step():
        optimizer.zero_grad()
        h = relu.forward(l1.forward(x))
        pred = l2.forward(h)
        loss = loss_fn.forward(pred, y)
        g = l2.backward(loss_fn.backward())
        g = relu.backward(g)
        l1.backward(g)
        optimizer.step()

    for _ in range(warmup):
        step()
    if torch_device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        step()
    if torch_device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / repeats
    return elapsed

for device_name in ('cpu', 'cuda'):
    if device_name == 'cuda' and not torch.cuda.is_available():
        print('GG1')
        continue
    dev = torch.device(device_name)
    for bs in (1, 16, 256):
        print(f'{device_name}, batch={bs}: {benchmark_minipy(dev, bs):.6f} сек/шаг')


cpu_parameter = torch.zeros(2, device='cpu')
if torch.cuda.is_available():
    cuda_parameter = torch.zeros(2, device='cuda')
    try:
        cpu_parameter + cuda_parameter
    except RuntimeError as error:
        print('GG')

## 3. Ручная INT8-квантизация

Симметричная квантизация входа и весов с накоплением произведений в `INT32`.

In [ ]:
class QuantizedLinear:
    def __init__(self, weight, bias=None, per_channel=False):
        weight = weight.detach().float().cpu()
        self.per_channel = per_channel

        if per_channel:
            weight_scale = weight.abs().amax(dim=1).clamp_min(1e-12) / 127
            weight_int8 = torch.round(weight / weight_scale[:, None]).clamp(-127, 127).to(torch.int8)
        else:
            weight_scale = weight.abs().max().clamp_min(1e-12) / 127
            weight_int8 = torch.round(weight / weight_scale).clamp(-127, 127).to(torch.int8)

        self.weight_int8 = weight_int8
        self.weight_scale = weight_scale
        self.bias = None if bias is None else bias.detach().float().cpu()

    def forward(self, x):
        x = x.detach().float().cpu()
        input_scale = x.abs().max().clamp_min(1e-12) / 127
        input_int8 = torch.round(x / input_scale).clamp(-127, 127).to(torch.int8)

        acc_int32 = input_int8.to(torch.int32) @ self.weight_int8.to(torch.int32).t()
        if self.per_channel:
            out = acc_int32.float() * (input_scale * self.weight_scale)[None, :]
        else:
            out = acc_int32.float() * (input_scale * self.weight_scale)

        if self.bias is not None:
            out = out + self.bias
        return out


weight = torch.tensor([[1.0, -0.5, 0.25], [100.0, -80.0, 20.0]])
bias = torch.tensor([0.1, -0.2])
x = torch.tensor([[0.1, 0.2, 0.3], [1.0, -1.0, 0.5]])

fp32_linear = torch.nn.Linear(3, 2)
with torch.no_grad():
    fp32_linear.weight.copy_(weight)
    fp32_linear.bias.copy_(bias)

fp32_output = fp32_linear(x)
for per_channel in (False, True):
    quantized = QuantizedLinear(weight, bias, per_channel=per_channel)
    output = quantized.forward(x)
    error = (fp32_output - output).abs()
    print(f'per_channel={per_channel}: mean={error.mean().item():.6f}, max={error.max().item():.6f}')

fp32_bytes = weight.numel() * weight.element_size()
for per_channel in (False, True):
    quantized = QuantizedLinear(weight, bias, per_channel=per_channel)
    int8_bytes = quantized.weight_int8.numel() * quantized.weight_int8.element_size()
    scale_bytes = quantized.weight_scale.numel() * quantized.weight_scale.element_size()
    print(f'per_channel={per_channel}: FP32={fp32_bytes} bytes, INT8+scales={int8_bytes + scale_bytes} bytes')

print('cast без масштабирования:', torch.zeros(3).to(torch.int8))

## Результаты

Сводные графики сохраняются в папку `results/` в формате PNG.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

quantization_results = {}
fp32_bytes = weight.numel() * weight.element_size()
for per_channel in (False, True):
    quantized = QuantizedLinear(weight, bias, per_channel=per_channel)
    output = quantized.forward(x)
    error = (fp32_output - output).abs()
    int8_bytes = quantized.weight_int8.numel() * quantized.weight_int8.element_size()
    scale_bytes = quantized.weight_scale.numel() * quantized.weight_scale.element_size()
    quantization_results["per-channel" if per_channel else "tensor"] = {
        "mean_error": error.mean().item(),
        "max_error": error.max().item(),
        "bytes": int8_bytes + scale_bytes,
    }

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses, color="#2563eb", linewidth=2)
axes[0].set(title="Миниатюрный PyTorch: loss", xlabel="Эпоха", ylabel="MSE")
axes[0].grid(alpha=0.25)

labels = list(quantization_results)
mean_errors = [quantization_results[label]["mean_error"] for label in labels]
stored_bytes = [quantization_results[label]["bytes"] for label in labels]
bars = axes[1].bar(labels, mean_errors, color=["#10b981", "#f59e0b"])
axes[1].set(title="INT8: средняя ошибка", ylabel="Absolute error")
axes[1].bar_label(bars, fmt="%.3f")
axes[1].grid(axis="y", alpha=0.25)

fig.suptitle("Сводка результатов", fontsize=14)
fig.tight_layout()
summary_path = results_dir / "results_summary.png"
fig.savefig(summary_path, dpi=180, bbox_inches="tight")
plt.close(fig)

plt.figure(figsize=(8, 4))
plt.plot(losses, color="#2563eb", linewidth=2)
plt.title("Кривая обучения")
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.grid(alpha=0.25)
loss_path = results_dir / "training_loss.png"
plt.savefig(loss_path, dpi=180, bbox_inches="tight")
plt.close()

print("Сохранено:", summary_path)
print("Сохранено:", loss_path)
print("Размер FP32:", fp32_bytes, "bytes")
for label, result in quantization_results.items():
    print(f"{label}: mean={result['mean_error']:.6f}, max={result['max_error']:.6f}, INT8={result['bytes']} bytes")